# FunSearch + RAG for the Capacitated Vehicle Routing Problem (CVRP)

## Overview

This notebook reproduces the **FunSearch + RAG** experiment pipeline for solving the **Capacitated Vehicle Routing Problem (CVRP)**.

### Task

The **CVRP** is a classic combinatorial optimisation problem:  
given a depot and a set of customers with known demands, find the minimum-cost set of vehicle routes such that every customer is visited exactly once and no route exceeds the vehicle's capacity.

### Method

We augment **FunSearch** — an evolutionary large-language-model (LLM) program search framework originally proposed by DeepMind — with a **Retrieval-Augmented Generation (RAG)** module:

| Component | Role |
|-----------|------|
| **FunSearch core** | Iteratively prompts an LLM to evolve a Python heuristic for CVRP, keeping the best programs in an island-based population |
| **RAG module** | Before each LLM call, retrieves the most relevant chunks from a curated knowledge corpus and injects them into the prompt, giving the model algorithmic context it would not otherwise have |
| **Evaluator** | Executes the generated heuristic on held-out CVRP benchmark instances (CVRPLIB Set-B) and returns a scalar score (negative total route distance) |

### Evaluation

Results are assessed on **CVRPLIB Set-B** benchmark instances. Key metrics reported:

- `best_score` — best negative total distance achieved across all islands
- `valid_eval_ratio` — fraction of LLM-generated programs that compiled and ran successfully
- `retrieval_events` — number of RAG retrievals performed
- `retrieval_mean_confidence` — mean semantic-similarity confidence of retrieved chunks

### Reproducibility

Each run clones a fixed GitHub branch, pins dependency versions, and logs all outputs to a timestamped directory under `results/experiments/`.

---
> **Budget control:** The default `VERIFICATION_BUDGET = 2` keeps cost low for smoke-testing.  
> Increase it (e.g. to `100`) for a full experiment run.

In [ ]:
import subprocess
import os

REPO_URL = "https://github.com/Zz1jd/CSProjectAI.git"
BRANCH = "RAG"
REPO_DIR = "/content/CSProjectAI"

if not os.path.exists(REPO_DIR):
    print("Cloning repository (first run)...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )
    print("Repository cloned successfully.")
else:
    print("Repository already exists – pulling latest changes...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)
    print("Repository updated successfully.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

git_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Branch : {BRANCH}")
print(f"Commit : {git_head}")

In [ ]:
!pip install -q vrplib numpy openai

## API Credentials

The pipeline calls two external APIs:

| Variable | Purpose |
|----------|---------|
| `FUNSEARCH_API_KEY` | LLM chat-completion API (OpenAI-compatible) |
| `FUNSEARCH_EMBEDDING_API_KEY` | Embedding API used by the RAG retriever |

**Option A – Set directly in the cell below** (easiest for Colab; keep the notebook private).  
**Option B – Mount Google Drive and load a `.env` file:**
```python
from google.colab import drive
drive.mount('/content/drive')
# Then place credentials in /content/drive/MyDrive/funsearch.env
```
**Option C – Use Colab Secrets** (Extras → Secrets) and read with `google.colab.userdata`.

> ⚠️ Never commit real API keys to a public repository.

In [ ]:
import os

os.environ["FUNSEARCH_API_KEY"] = "sk-vWpzPgcJaoamJOr998VvL5H4Z2uTt6jNmPk0SftpmCQJYZ5C"
os.environ["FUNSEARCH_EMBEDDING_API_KEY"] = "sk-jwbxcbszdqdinhqofxikohzyjisdvwnkljbrzkfqufuxcbyy"

# Confirm registration without printing the key values.
print("FUNSEARCH_API_KEY set         :", bool(os.environ.get("FUNSEARCH_API_KEY")))
print("FUNSEARCH_EMBEDDING_API_KEY set:", bool(os.environ.get("FUNSEARCH_EMBEDDING_API_KEY")))

## Dataset & Problem Description

### Capacitated Vehicle Routing Problem (CVRP)

The **CVRP** asks: given a depot node `0` and `n` customer nodes, each with a demand $d_i$, find a minimum-cost set of routes for a fleet of vehicles with uniform capacity $Q$ such that:
- Every customer is visited exactly once.
- The total demand on each route does not exceed $Q$.
- Every route starts and ends at the depot.

Formally, the objective is:

$$\min \sum_{k} \sum_{(i,j) \in \text{route}_k} c_{ij}$$

where $c_{ij}$ is the Euclidean distance between nodes $i$ and $j$.

### Benchmark: CVRPLIB Set-B

We evaluate on **CVRPLIB Set-B** (Christofides & Eilon, 1969) — 23 standardised instances ranging from 31 to 78 customers. The dataset is stored in the repository under `cvrplib/setB/` in the standard `.vrp` / `.sol` format. It is loaded by `dataset.py → load_cvrp_dataset()`.

The evaluator scores a candidate heuristic by executing it on these instances and returning the **negative total distance** (higher is better, so FunSearch maximises this).

### Knowledge Corpus for RAG

The RAG module draws from a curated corpus (`corpus/`) of chunked algorithmic knowledge — VRP heuristics, savings algorithms, and neighbourhood-search descriptions — stored as text chunks. At each sampling step the retriever finds the most semantically similar chunks to the current prompt context and injects them as additional context for the LLM.

In [ ]:
import vrplib
from dataset import load_cvrp_dataset

DATASET_PATH = "cvrplib/setB"
cvrp_dataset = load_cvrp_dataset(DATASET_PATH)
instances = cvrp_dataset["B"]

print(f"Loaded {len(instances)} CVRPLIB Set-B instances from '{DATASET_PATH}'\n")
print(f"{'Instance':<20} {'Nodes':>6} {'Capacity':>10} {'Total demand':>14} {'Vehicles (k)':>14}")
print("─" * 70)
for name, inst in sorted(instances.items()):
    n_nodes   = inst["dimension"] - 1       # exclude depot
    capacity  = inst["capacity"]
    total_dem = int(inst["demand"][1:].sum())
    k_label   = name.split("-")[-1]         # e.g. "k5"
    print(f"{name:<20} {n_nodes:>6} {capacity:>10} {total_dem:>14} {k_label:>14}")

### Visualising a Problem Instance

Before running the search, it is helpful to visualise the structure of one benchmark instance. Below we plot the node coordinates for `B-n31-k5` (the smallest instance). The depot is marked with a red star; customer circles are sized proportionally to their demand, making capacity constraints immediately visible.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

VIZ_INSTANCE = "B-n31-k5"
inst = instances[VIZ_INSTANCE]

coords  = np.array(inst["node_coord"])  # shape (n+1, 2)
demands = np.array(inst["demand"])

depot_xy    = coords[0]
customer_xy = coords[1:]
cust_dem    = demands[1:]

fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    customer_xy[:, 0], customer_xy[:, 1],
    s=cust_dem * 6,       # marker area ∝ demand
    c=cust_dem,
    cmap="Blues",
    edgecolors="steelblue",
    linewidths=0.6,
    zorder=3,
    label="Customer (size ∝ demand)",
)
plt.colorbar(sc, ax=ax, label="Demand")

ax.scatter(depot_xy[0], depot_xy[1],
           marker="*", s=300, c="red", zorder=5, label="Depot")

ax.set_title(
    f"CVRP Instance: {VIZ_INSTANCE}\n"
    f"{len(customer_xy)} customers · capacity {inst['capacity']}",
    fontsize=12,
)
ax.set_xlabel("x-coordinate")
ax.set_ylabel("y-coordinate")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Running the Experiment

With the dataset loaded and inspected, we now run the **FunSearch + RAG** pipeline. The experiment follows this flow:

```
Dataset (CVRPLIB Set-B)
        │
        ▼
 FunSearch Core  ◄──── Island Population (evolutionary program database)
        │
        ├── Prompt construction (Chain-of-Thought template)
        │         │
        │         └── RAG Retriever ──► corpus/ (chunked VRP knowledge)
        │
        ├── LLM call  ──► generated Python heuristic
        │
        └── Evaluator ──► score (negative total distance)
                │
                └── best score updated in island population
```

The three cells below handle:
1. **Imports** — loads the experiment runner helpers from `scripts/run_rag_eval.py`
2. **Configuration** — sets budget, model, and seed
3. **Run** — executes the pipeline and writes logs to `results/experiments/`

### Configuration

`VERIFICATION_BUDGET` controls the number of LLM sampling steps FunSearch performs.  
Set it to `2` for a quick smoke-test, or `100` for a full experiment run.

| Parameter | Description |
|-----------|-------------|
| `budget` | Maximum number of LLM samples (≈ API calls) |
| `run_mode` | `"stage_eval"` — evaluate each generated program on all dataset instances |
| `seed` | Global random seed for reproducibility |
| `model_name` | OpenAI-compatible model identifier sent to the API |
| `result_label` | Suffix for the timestamped output directory |

In [ ]:
from pathlib import Path

from scripts.run_rag_eval import ExperimentRunConfig
from scripts.run_rag_eval import ModelSpec
from scripts.run_rag_eval import run_rag_eval

print("Imports OK – experiment helpers loaded.")

In [ ]:
VERIFICATION_BUDGET = 2  # increase to 100 for a full run

experiment_config = ExperimentRunConfig(budget=VERIFICATION_BUDGET)

llm_model = ModelSpec(
    model_name="gpt-3.5-turbo",
    result_label="test_gpt_3_5_turbo",
)

print(f"Budget   : {experiment_config.budget} samples")
print(f"Run mode : {experiment_config.run_mode}")
print(f"Seed     : {experiment_config.seed}")
print(f"Model    : {llm_model.model_name}")
print(f"Label    : {llm_model.result_label}")

In [ ]:
print("Starting experiment… (this may take a few minutes for budget > 2)")
experiment_summary = run_rag_eval(
    experiment_config=experiment_config,
    model_spec=llm_model,
)
print("Experiment complete.")
experiment_summary

### Results

After the run completes, the cell below prints the key metrics.

| Metric | Meaning |
|--------|---------|
| `best_score` | Best negative total distance found — less negative means shorter routes |
| `valid_eval_ratio` | Fraction of LLM-generated programs that compiled and produced a finite score |
| `retrieval_events` | Total number of RAG corpus retrievals performed |
| `retrieval_mean_confidence` | Mean cosine-similarity of retrieved chunks — higher means more on-topic context |

All outputs are also saved to `results/experiments/<timestamp>_<label>_rag/` as `rag.txt` (raw log) and `rag_summary.json` (structured summary).

In [ ]:
import json

summary_path = Path(experiment_summary["summary_path"])
run_result   = experiment_summary["run"]

print("─" * 50)
print(f"  Output directory  : {experiment_summary['experiment_dir']}")
print(f"  Summary JSON      : {summary_path}")
print(f"  Raw log           : {run_result['log_path']}")
print("─" * 50)
print(f"  best_score               : {run_result['best']}")
print(f"  valid_eval_ratio         : {run_result['valid_eval_ratio']}")
print(f"  retrieval_events         : {run_result['retrieval_events']}")
print(f"  retrieval_mean_confidence: {run_result['retrieval_mean_confidence']}")
print("─" * 50)
print(json.dumps(experiment_summary, indent=2))

## Conclusion

In this notebook we demonstrated a complete end-to-end run of **FunSearch augmented with Retrieval-Augmented Generation (RAG)** applied to the **Capacitated Vehicle Routing Problem (CVRP)**.

### What we did

| Step | Description |
|------|-------------|
| **1. Setup** | Cloned the repository from GitHub, ensuring a reproducible, pinned code version |
| **2. Dependencies** | Installed `vrplib`, `numpy`, and `openai` — the minimal dependency set |
| **3. Dataset** | Loaded **CVRPLIB Set-B** (up to 78 customers) and visualised one instance to build problem intuition |
| **4. Experiment** | Ran FunSearch with RAG: the LLM iteratively evolved a Python CVRP heuristic, guided by semantically relevant algorithmic knowledge retrieved from the corpus at each step |
| **5. Results** | Inspected `best_score`, `valid_eval_ratio`, and retrieval diagnostics |

### Key takeaways

- **RAG improves prompt quality**: by injecting domain knowledge (VRP heuristics, savings algorithms) directly into the LLM prompt, the model can produce higher-quality routing heuristics from fewer samples.
- **FunSearch is sample-efficient**: evolutionary island-based search keeps a diverse population of programs, avoiding local optima.
- **Reproducibility**: all results are logged under a timestamped directory with a full JSON config snapshot.

### Next steps

- Increase `VERIFICATION_BUDGET` to `100` or more for publication-quality results.
- Try other models via `ModelSpec(model_name=...)`.
- Extend the corpus with more VRP literature to further improve RAG quality.